# Data Preparation Research

## Данные
- `data/raw/dataset.csv` — исходный датасет
- `docs/data_dictionary.xlsx` — описание столбцов с указанием целевых типов

### 1. Импорт и загрузка

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/dataset.csv')

### 2. Первичный осмотр данных: shape, info

In [ ]:
print(f'Размер датасета: {df.shape}')

#создаем файл со сводной информацией о датасете: столбцы, типы данных по столбцам, пропуски, кол-во уникальных значений (см. data/data_dictionary.xlsx)
info_df = pd.DataFrame({
    'Столбец': df.columns,
    'Тип данных': df.dtypes.values,
    'Пропуски': df.isnull().sum().values,
    'Пропуски %': (df.isnull().sum().values / len(df) * 100).round(2),
    'Заполнено': df.notnull().sum().values,
    'Уникальных': df.nunique().values
})
info_df.to_excel('data_overview.xlsx', index=False)

### 3. Проверка на дублирование

In [ ]:
pairs = [
    ('lead_created_at', 'lead_Дата создания сделки'),
    ('handed_to_delivery_ts', 'lead_Дата перехода Передан в доставку'),
    ('returned_ts', 'lead_Дата возврата посылки на склад'),
    ('sale_ts', 'sale_date'),
    ('lead_source', 'lead_Источник'),
    ('lead_responsible_user_id', 'contact_responsible_user_id'),
    ('lead_LTV', 'contact_LTV'),
    ('lead_group', 'lead_group_id'),
    ('lead_status_id', 'current_status_id'),
    ('lead_Сумма наложенного платежа (руб)', 'lead_Объявленная ценность (руб)'),
    ('lead_Сумма заказа', 'lead_price'),
    ('lead_LEADQUALIFYCATION', 'lead_Квалификация лида'),
    ('lead_Масса (гр)', 'lead_Вес (грамм)*'),
    ('lead_Линейная ширина (см)', 'lead_Ширина'),
    ('lead_Линейная высота (см)', 'lead_Высота'),
    ('lead_Линейная длина (см)', 'lead_Длина'),
]

results = []

for col1, col2 in pairs:
    both = df[[col1, col2]].dropna()
    if len(both) > 0:
        match = (both[col1].astype(str) == both[col2].astype(str)).sum()
        results.append((col1, col2, match/len(both)))

pd.DataFrame(results, columns=['col1', 'col2', 'match_rate']).sort_values(by='match_rate', ascending=False)

### Выводы

- Обнаружены полные или почти полные дубли столбцов
- Например:
    - 'current_status_id' = lead_status_id
    - lead_Сумма наложенного платежа (руб) ~ lead_Объявленная ценность (совпадает на 99.9%)
- Эти столбцы будут удалены при запуске data_preparation.py

### Проверка возможности приведения типов к целевым без ошибок и потери значений

In [ ]:
dd = pd.read_excel('docs/data_dictionary.xlsx')

needs_fix = dd[dd['Исходный тип данных'] != dd['Целевой тип данных']]
needs_fix.head()

grouped = needs_fix.groupby(['Исходный тип данных', 'Целевой тип данных'])
#пример проверки; если есть сомнения - проверяем таким же образом и другие типы конвертации
str_to_float_cols = [
    col for (src, tgt), group in grouped
    if src == 'str' and tgt == 'float64'
    for col in group['Столбец']
]

for col in str_to_float_cols[:3]:
    converted = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
    print(col, df[col].notna().sum() - converted.notna().sum())

### Вывод

- Потерь при конвертации нет
- Формат данных корректный
- Конвертацию можно выполнять безопасно